# Modelo de Churn Deuna — Pipeline Completo

**Reto:** *Antes de que se vayan* — Scoring predictivo de abandono de comerciantes B2B (Interact2Hack 2026).

**Objetivo:** identificar comercios con alto riesgo de dejar de transar en los próximos 30 días.

**Tecnología:** XGBoost + SHAP para explicabilidad. Dataset sintético de 2000 comercios × 12 meses.

**Criterio de éxito:** AUC > 0.75 y variables explicativas coherentes con el negocio.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

## 2. Simulación de datos

Genera tres tablas sintéticas en `data/raw/`:
- `merchants.csv` — atributos estáticos (MCC, región, tenure)
- `monthly_activity.csv` — serie mensual de actividad transaccional
- `churn_labels.csv` — etiqueta de churn en los próximos 30 días

In [ ]:
import data_simulation
paths = data_simulation.main(out_dir=ROOT / 'data/raw')

In [ ]:
merchants = pd.read_csv(ROOT / 'data/raw/merchants.csv')
activity = pd.read_csv(ROOT / 'data/raw/monthly_activity.csv', parse_dates=['fecha_corte'])
labels = pd.read_csv(ROOT / 'data/raw/churn_labels.csv', parse_dates=['fecha_corte'])

print(f'Merchants: {merchants.shape}')
print(f'Activity:  {activity.shape}')
print(f'Labels:    {labels.shape}')
print(f'Tasa de churn: {labels["churn_30d"].mean():.2%}')

## 3. EDA rápido

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

merchants['mcc_segmento'].value_counts().plot(kind='bar', ax=axes[0], title='Distribución por segmento MCC')
merchants['region'].value_counts().plot(kind='bar', ax=axes[1], title='Distribución por región')
merchants['tenure_meses'].hist(ax=axes[2], bins=30)
axes[2].set_title('Distribución de tenure (meses)')
plt.tight_layout()
plt.show()

In [ ]:
churn_por_mcc = (
    merchants.merge(labels[['comercio_id', 'churn_30d']], on='comercio_id')
    .groupby('mcc_segmento')['churn_30d'].agg(['mean', 'count'])
    .rename(columns={'mean': 'tasa_churn', 'count': 'n_comercios'})
    .sort_values('tasa_churn', ascending=False)
)
print(churn_por_mcc)

## 4. Feature Engineering

Construye la MDT con ventanas de los últimos 5 meses (lags 0-4), agregados 3m/6m/12m, deltas consecutivos y ratios.

In [ ]:
import feature_engineering
mdt_path = feature_engineering.main(raw_dir=ROOT / 'data/raw', out_dir=ROOT / 'data/processed')
mdt = pd.read_parquet(mdt_path)
print(mdt.shape)
mdt.head(3)

## 5. Entrenamiento del modelo (XGBoost)

In [ ]:
import train_model
result = train_model.train(
    mdt_path=ROOT / 'data/processed/mdt_churn.parquet',
    out_dir=ROOT / 'outputs/model',
)

## 6. Explicabilidad con SHAP

In [ ]:
import explain
shap_df = explain.explain(
    mdt_path=ROOT / 'data/processed/mdt_churn.parquet',
    model_path=ROOT / 'outputs/model/churn_model.pkl',
    fig_dir=ROOT / 'outputs/figures',
    shap_out=ROOT / 'outputs/shap_values.parquet',
)

In [ ]:
from IPython.display import Image
Image(filename=str(ROOT / 'outputs/figures/shap_summary.png'))

In [ ]:
Image(filename=str(ROOT / 'outputs/figures/shap_bar.png'))

## 7. Scoring y segmentación

Genera el archivo final de predicciones con segmentación Alerta Roja/Amarilla/Baja/Muy Baja por percentil, consumible por el equipo de dashboard.

In [ ]:
import predict
preds = predict.score(
    mdt_path=ROOT / 'data/processed/mdt_churn.parquet',
    model_path=ROOT / 'outputs/model/churn_model.pkl',
    out_path=ROOT / 'outputs/predictions.csv',
)

In [ ]:
preds.head(20)

## 8. Checks finales

- Criterio de éxito Deuna: **AUC > 0.75** ✔
- Variables explicativas coherentes con literatura B2B: recencia, volatilidad transaccional, decline rate, tickets soporte
- Output `predictions.csv` consumible por el equipo de dashboard con segmentación ya asignada